<a href="https://colab.research.google.com/github/eduardokern/ML/blob/face_detection/notebooks/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Criando um sistema de reconhecimento facial do zero</h1>

O objetivo principal deste projeto é trabalhar com as bibliotecas e frameworks estudados e analisados em nossas aulas. Neste sentido, a proposta padrão envolve um sistema de detecção e reconhecimento de faces, utilizando o framework TensorFlow em conjuntos com as bibliotecas que o projetista julgue necessárias, de forma ilimitada.  

Por meio da Figura 1 é possível visualizar o resultado esperado para o modelo proposto, devendo detectar e reconhecer mais de uma face ao mesmo tempo.  
Para isso você deve:

1. Utilizar uma rede de detecção treinada para detectar faces.
2. Utilizar uma rede de classificação para classificar a face detectada.

![Figura 1: Detecção e reconhecimento facial.](https://drive.google.com/uc?export=view&id=1nFCZd-FR0jIYAvDbDbVpDt6ansjEkLIA)

Para realizar este projeto, você pode utilizar os seguintes trabalhos de referência:
Detecção Facial:
https://colab.research.google.com/drive/1QnC7lV7oVFk5OZCm75fqbLAfD9qBy9bw?usp=sharing

Detecção e classificação de objetos:  
https://colab.research.google.com/drive/1xdjyBiY75MAVRSjgmiqI7pbRLn58VrbE?usp=sharing

<h3>Mount Google Drive to get datasets</h3>

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

datasets_path = '/content/drive/MyDrive/Colab/ML/datasets'

Mounted at /content/drive


<h3>Required imports</h3>

In [ ]:
import keras
import matplotlib.pyplot as plt
import math
import numpy as np
import os
import random
from keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout
from keras.models import Sequential
from keras.preprocessing import image
from PIL import ImageDraw

<h3>Helper functions</h3>

In [ ]:
# helper function to load image and return it and input vector
def get_image(path):
  try:
    img = image.load_img(path, target_size=(224, 224), keep_aspect_ratio=True)
    data = image.img_to_array(img)
    data = np.expand_dims(data, axis=0)
    return img, data
  except Exception as e:
    print(f'Error loading: {path}, {e}')
    return None, None


In [ ]:
# Getting only images that detect a single person on an image.
# Not implemented SSD
def load_detection_datasets(path, store_image):
    dataset = []
    with open(f'{path}/_annotations.csv') as file:
        file.readline()
        lines = file.readlines()

    single = {}
    for line in lines:
        values = line.replace('\n', '').split(',')
        if len(values) > 1:
            filename = f'{path}/{values[0]}'
            if os.path.exists(filename):
                # label = values[3]
                width = int(values[1])
                height = int(values[2])
                xmin = float(values[4])
                ymin = float(values[5])
                xmax = float(values[6])
                ymax = float(values[7])

                single.setdefault(filename, []).append([xmin/width, ymin/height, xmax/width, ymax/height])

    single = [(filename, bbox[0]) for filename, bbox in single.items() if len(bbox) == 1]

    if len(single) > 1000:
        random.shuffle(single)
        single = single[0:1000]

    for filename, bbox in single:
        image_data, data = get_image(filename)
        dataset.append(
            {
                'x': np.array(data[0], dtype='float32') / 255.,
                'y': np.array(bbox, dtype='float32'),
                'image': image_data if store_image else None
            }
        )

    return dataset

In [ ]:
# Helper function to load faces of the persons to classify
def load_classes_datasets(categories_path):
    datasets = {}
    categories_paths = [x[0] for x in os.walk(categories_path) if x[0]][1:]
    for c, cat_path in enumerate(categories_paths):
        cat_path = cat_path.replace('\\', '/')
        category = cat_path.split('/')[-1]
        datasets[category] = []
        print(f'Loading {category} from {cat_path}')
        images = [os.path.join(dp, f) for dp, dn, filenames
                in os.walk(cat_path) for f in filenames
                if os.path.splitext(f)[1].lower() in ['.jpg','.png','.jpeg', '.jfif']]
        for img_path in images:
            img, data = get_image(img_path)
            if img is not None:
                datasets[category].append({
                    'x': np.array(data[0], dtype='float32') / 255.,
                    'y': c,
                    'image': img
                })

    num_classes = len(datasets)
    for dataset in datasets.values():
        for img in dataset:
            img['y'] = keras.utils.to_categorical(img['y'], num_classes)

    train_split = 0.7
    val_split = 0.2

    train = []
    val = []
    test = []
    for images in datasets.values():
        random.shuffle(images)
        idx_val = int(train_split * len(images))
        idx_test = int((train_split + val_split) * len(images))
        train += images[:idx_val]
        val += images[idx_val:idx_test]
        test += images[idx_test:]

    return train, val, test, list(datasets.keys())

In [ ]:
# Helper function to show a list of images
def show_image(images, rows, columns):
    fig, axs = plt.subplots(rows, columns, figsize=(20, 20))
    i = 0
    max = len(images)
    for row in range(rows):
        for col in range(columns):
            ax = axs[row][col] if rows > 1 and columns > 1 else axs[i]
            ax.axis('off')
            if i < max:
                img = images[i]
                ax.imshow(img)
                i += 1

In [ ]:
# Helper function to draw the bounding box and the label in the predicted image
def draw_info(img, bbox, label):
    draw = ImageDraw.Draw(img)
    draw.rectangle(((bbox[0], bbox[1]), (bbox[2], bbox[3])), outline = 'green')
    draw.text((bbox[0], bbox[1]), label, 'green')

<h3>FaceDetection class</h3>

It contais a CNN to detect all faces and their bbox and a VGG16 neural network to classify the faces detected.

Both NN can be trained with different datasets.

In [ ]:
class FaceDetection:
    def __init__(
        self, detection_train_data, detection_val_data, path, epochs, batch_size
    ):
        self._path = path
        self._weights_filename = 'face_detection.weights.h5'
        self._weights_class_filename = 'classification.weights.h5'
        self._num_classes_filename = 'classes.txt'

        self._detection_model()
        weights_filename = f'{path}/{self._weights_filename}'
        if (os.path.exists(weights_filename)):
            self._detection_model.load_weights(weights_filename)
        else:
            self._train_detection(detection_train_data, detection_val_data, epochs, batch_size)
            self._detection_model.save_weights(weights_filename)

        weights_class_filename = f'{path}/{self._weights_class_filename}'
        num_classes_filename = f'{path}/{self._num_classes_filename}'
        if (os.path.exists(weights_class_filename)) and (os.path.exists(num_classes_filename)):
            with open(num_classes_filename) as file:
                num_classes = int(file.readline())

            self._classification_model(num_classes)
            self._classification_model.load_weights(weights_class_filename)


    def _detection_model(self):
        # Transfer learning
        # vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
        # vgg.trainable = False

        # flatten = Flatten(name='detection_output')(vgg.output)
        # bbox = Dense(128, activation='relu')(flatten)
        # bbox = Dense(64, activation='relu')(bbox)
        # bbox = Dense(32, activation='relu')(bbox)
        # bbox = Dense(4, activation='sigmoid', name='bbox')(bbox)

        # self._detection_model = Model(inputs=vgg.input, outputs=bbox)

        # From scratch
        self._detection_model = Sequential()
        self._detection_model.add(Input(shape=(224, 224, 3)))
        self._detection_model.add(Conv2D(32, (3, 3), activation='relu'))
        self._detection_model.add(Conv2D(64, (3, 3), activation='relu'))
        self._detection_model.add(Flatten())
        self._detection_model.add(Dense(128, activation='relu'))
        # Output layer for bounding box coordinates
        self._detection_model.add(Dense(4, activation='linear'))


    def _train_detection(self, train_data, val_data, epochs, batch_size):
        self._detection_model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])
        self._detection_model.fit(
            train_data[0],
            train_data[1],
            validation_data=val_data,
            epochs=epochs,
            batch_size=batch_size)


    def detect(self, image_data):
        data = image.img_to_array(image_data)
        data = np.expand_dims(data, axis=0)
        data = np.array(data, dtype='float32') / 255.
        result = self._detection_model.predict(data)[0]
        if result[0] > result[2]:
            result[0], result[2] = result[2], result[0]
        if result[1] > result[3]:
            result[1], result[3] = result[3], result[1]
        w, h = image_data.size
        return [result[0]*w, result[1]*h, result[2]*w, result[3]*h]


    def _classification_model(self, num_classes):
        vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
        inp = vgg.input

        # make a new softmax layer with num_classes neurons
        new_classification_layer = Dense(num_classes, activation='softmax')

        # connect our new layer to the second to last layer in VGG, and make a reference to it
        out = new_classification_layer(vgg.layers[-2].output)

        # create a new network between inp and out
        self._classification_model = Model(inp, out)

        # make all layers untrainable by freezing weights (except for last layer)
        for l, layer in enumerate(self._classification_model.layers[:-1]):
            layer.trainable = False

        # ensure the last layer is trainable/not frozen
        for _, layer in enumerate(self._classification_model.layers[-1:]):
            layer.trainable = True

        self._classification_model.compile(
            loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


    def train_classification(self, num_classes, train_data, val_data, epochs, batch_size):
        self._classification_model(num_classes)

        self._classification_model.fit(
            train_data[0],
            train_data[1],
            validation_data=val_data,
            epochs=epochs,
            batch_size=batch_size)

        with open(f'{self._path}/{self._num_classes_filename}', 'w') as file:
            file.write(f'{num_classes}')

        self._classification_model.save_weights(f'{self._path}/{self._weights_class_filename}')


    def predict(self, img):
        bbox = self.detect(img)
        face = img.crop(bbox)
        face = face.resize((224, 224))
        data = image.img_to_array(face)
        data = np.expand_dims(data, axis=0)
        data = np.array(data, dtype='float32') / 255.
        return self._classification_model.predict(data), bbox

Load face detection dataset to train the face detection CNN to identify all faces and their bbox in an image.

In [ ]:
# load detection dataset
detection_dataset_path = f'{datasets_path}/FaceDetection'
detection_train = load_detection_datasets(f'{detection_dataset_path}/train', False)
detection_valid = load_detection_datasets(f'{detection_dataset_path}/valid', False)
detection_test = load_detection_datasets(f'{detection_dataset_path}/test', True)

x_detection_train, y_detection_train = np.array([t["x"] for t in detection_train]), np.array([t["y"] for t in detection_train])
x_detection_valid, y_detection_valid = np.array([t["x"] for t in detection_valid]), np.array([t["y"] for t in detection_valid])

In [ ]:
# load classification dataset
class_dataset_path = f'{datasets_path}/big3'
class_train, class_val, class_test, classes = load_classes_datasets(class_dataset_path)

x_class_train, y_class_train = np.array([t["x"] for t in class_train]), np.array([t["y"] for t in class_train])
x_class_valid, y_class_valid = np.array([t["x"] for t in class_val]), np.array([t["y"] for t in class_val])

In [ ]:
# Initializing face detector and setting up detection network
detector = FaceDetection(
   (x_detection_train, y_detection_train),
   (x_detection_valid, y_detection_valid),
   detection_dataset_path,
   20,
   32)

# Training face classification
num_classes = len(classes)
if (not os.path.exists(f'{detection_dataset_path}/classes.txt')):
    detector.train_classification(
        num_classes,
        (x_class_train, y_class_train),
        (x_class_valid, y_class_valid),
        20,
        32)

In [ ]:
# Testing detection
imgs = []
for test in detection_test[:len(detection_test)/2]:
    img = test['image']
    bbox = detector.detect(img)
    draw_info(img, bbox, 'Face')
    imgs.append(img)

In [ ]:
# Testing face classification
for test in class_test:
    img = test['image']
    prediction, bbox = detector.predict(img)

    cat = 0
    max = 0
    for i, result in enumerate(prediction[0]):
        if result > max:
            cat = i
            max = result

    draw_info(img, bbox, f'{classes[i]} : {max*100:.1f}%')
    imgs.append(img)

In [ ]:
# Showing both results, face detection and classification
cols = 6
rows = math.ceil(len(imgs) / 6.)
show_image(imgs, rows, cols)